In [1]:
import os, re, json
import numpy as np
import pandas as pd

DATASET_DIR = "/kaggle/input/cafa-6-protein-function-prediction"
TRAIN_DIR = os.path.join(DATASET_DIR, "Train")
TEST_DIR  = os.path.join(DATASET_DIR, "Test")

TRAIN_FASTA = os.path.join(TRAIN_DIR, "train_sequences.fasta")
TRAIN_TERMS = os.path.join(TRAIN_DIR, "train_terms.tsv")
TRAIN_TAXON = os.path.join(TRAIN_DIR, "train_taxonomy.tsv")

TEST_FASTA  = os.path.join(TEST_DIR, "testsuperset.fasta")
SAMPLE_SUB  = os.path.join(DATASET_DIR, "sample_submission.tsv")

**Helpers (FASTA + ID normalize)******

In [2]:
def read_fasta_to_df(path: str) -> pd.DataFrame:
    ids, seqs = [], []
    cur_id, cur_seq = None, []
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith(">"):
                if cur_id is not None:
                    ids.append(cur_id)
                    seqs.append("".join(cur_seq))
                cur_id = line[1:].split()[0]
                cur_seq = []
            else:
                cur_seq.append(line)
    if cur_id is not None:
        ids.append(cur_id)
        seqs.append("".join(cur_seq))
    df = pd.DataFrame({"protein_id": ids, "sequence": seqs})
    df["length"] = df["sequence"].str.len()
    return df

def normalize_protein_id(x: str) -> str:
    x = str(x).strip().split()[0]
    # UniProt style: sp|ACC|NAME or tr|ACC|NAME
    if "|" in x:
        parts = x.split("|")
        if len(parts) >= 2:
            x = parts[1]
        else:
            x = parts[-1]
    # remove version suffix like ABC123.1
    if "." in x:
        left, right = x.rsplit(".", 1)
        if right.isdigit():
            x = left
    return x

**Load train/test + labels**

In [3]:
train_seq = read_fasta_to_df(TRAIN_FASTA)
test_seq  = read_fasta_to_df(TEST_FASTA)

train_seq["protein_id"] = train_seq["protein_id"].map(normalize_protein_id)
test_seq["protein_id"]  = test_seq["protein_id"].map(normalize_protein_id)

train_terms = pd.read_csv(TRAIN_TERMS, sep="\t")
train_terms_std = train_terms.rename(columns={"EntryID":"protein_id","term":"go_id","aspect":"aspect"}).copy()
train_terms_std["protein_id"] = train_terms_std["protein_id"].map(normalize_protein_id)

train = train_seq.copy()
test  = test_seq.copy()

print("train:", train.shape, "| test:", test.shape, "| terms:", train_terms_std.shape)
print("train_terms_std cols:", train_terms_std.columns.tolist())


train: (82404, 3) | test: (224309, 3) | terms: (537027, 3)
train_terms_std cols: ['protein_id', 'go_id', 'aspect']


In [4]:
with open(SAMPLE_SUB, "r", encoding="utf-8") as f:
    for _ in range(3):
        print("sample protein_id:", f.readline().strip().split("\t")[0])
print("test protein_id:", test["protein_id"].iloc[0])

sample protein_id: A0A0C5B5G6
sample protein_id: A0A0C5B5G6
sample protein_id: A0A0C5B5G6
test protein_id: A0A0C5B5G6


**Create train/val split with duplicate-sequence safety**

In [5]:
SEED = 42
VAL_FRAC = 0.10

train_ids_path = "/kaggle/working/train_ids.txt"
val_ids_path   = "/kaggle/working/val_ids.txt"

if os.path.exists(train_ids_path) and os.path.exists(val_ids_path):
    train_ids = pd.read_csv(train_ids_path, header=None)[0].astype(str).tolist()
    val_ids   = pd.read_csv(val_ids_path,   header=None)[0].astype(str).tolist()
    print("Loaded existing split:", len(train_ids), len(val_ids))
else:
    seq_to_rep = train.groupby("sequence")["protein_id"].min().rename("rep_id").reset_index()
    train_rep  = train.merge(seq_to_rep, on="sequence", how="left")

    rep_ids = train_rep["rep_id"].drop_duplicates().values
    rng = np.random.default_rng(SEED)
    rng.shuffle(rep_ids)

    n_val = int(len(rep_ids) * VAL_FRAC)
    val_rep_ids = set(rep_ids[:n_val])
    train_rep_ids = set(rep_ids[n_val:])

    val_ids   = train_rep.loc[train_rep["rep_id"].isin(val_rep_ids), "protein_id"].unique().tolist()
    train_ids = train_rep.loc[train_rep["rep_id"].isin(train_rep_ids), "protein_id"].unique().tolist()

    pd.Series(train_ids).to_csv(train_ids_path, index=False, header=False)
    pd.Series(val_ids).to_csv(val_ids_path, index=False, header=False)

    print("Created split:", len(train_ids), len(val_ids))
    print("protein overlap:", len(set(train_ids) & set(val_ids)))

Created split: 74150 8254
protein overlap: 0


**Build GO label vocab + pid→label list maps**

In [6]:
train_terms_train = train_terms_std[train_terms_std["protein_id"].isin(train_ids)][["protein_id","go_id"]].copy()
go_vocab = sorted(train_terms_train["go_id"].unique())
go2i = {go:i for i,go in enumerate(go_vocab)}
i2go = {i:go for go,i in go2i.items()}
n_labels = len(go_vocab)
print("n_labels:", n_labels)

def make_pid2labels(pids):
    df = train_terms_std[train_terms_std["protein_id"].isin(pids)][["protein_id","go_id"]].copy()
    df = df[df["go_id"].isin(go2i)]
    df["y"] = df["go_id"].map(go2i).astype(int)
    return df.groupby("protein_id")["y"].apply(list).to_dict()

train_pid2labels = make_pid2labels(set(train_ids))
val_pid2labels   = make_pid2labels(set(val_ids))

print("train proteins w/labels:", len(train_pid2labels))
print("val proteins w/labels:", len(val_pid2labels))

path = "/kaggle/working/go_vocab.json"
with open(path, "w", encoding="utf-8") as f:
    json.dump(go_vocab, f)

print("saved to:", path)
print("exists:", os.path.exists(path))
print("size bytes:", os.path.getsize(path))


n_labels: 25576
train proteins w/labels: 74150
val proteins w/labels: 8243
saved to: /kaggle/working/go_vocab.json
exists: True
size bytes: 358064


**Train CNN**

In [7]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

MAX_LEN = 512
BATCH_SIZE = 128
NUM_WORKERS = 2
EPOCHS = 3
LR = 1e-3

PAD = 0
UNK = 1

lut = np.full(256, UNK, dtype=np.uint8)
alphabet = "ACDEFGHIKLMNPQRSTVWY"
for i, aa in enumerate(alphabet, start=2):
    lut[ord(aa)] = i

def encode_fixed(seq: str) -> np.ndarray:
    b = np.frombuffer(seq.encode("ascii", "ignore"), dtype=np.uint8)
    x = lut[b]
    out = np.full((MAX_LEN,), PAD, dtype=np.uint8)
    n = min(len(x), MAX_LEN)
    out[:n] = x[:n]
    return out

class ProteinDataset(Dataset):
    def __init__(self, df, pid2labels):
        self.pids = df["protein_id"].astype(str).tolist()
        self.pid2labels = pid2labels
        self.X = np.stack([encode_fixed(s) for s in df["sequence"].tolist()], axis=0)

    def __len__(self):
        return len(self.pids)

    def __getitem__(self, idx):
        pid = self.pids[idx]
        x = self.X[idx]
        labs = self.pid2labels.get(pid, [])
        return x, labs

def collate_batch(batch):
    xs, labs_list = zip(*batch)
    X = torch.from_numpy(np.stack(xs, axis=0)).long()
    Y = torch.zeros((len(xs), n_labels), dtype=torch.float32)
    for i, labs in enumerate(labs_list):
        if len(labs) > 0:
            Y[i, labs] = 1.0
    return X, Y

train_df = train.loc[train["protein_id"].isin(train_ids), ["protein_id","sequence"]].copy()
val_df   = train.loc[train["protein_id"].isin(val_ids),   ["protein_id","sequence"]].copy()

train_loader = DataLoader(
    ProteinDataset(train_df, train_pid2labels),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_batch
)

class CNNClassifier(nn.Module):
    def __init__(self, vocab_size=256, emb_dim=128, ch=256, n_labels=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.conv3 = nn.Conv1d(emb_dim, ch, kernel_size=3, padding=1)
        self.conv5 = nn.Conv1d(emb_dim, ch, kernel_size=5, padding=2)
        self.conv7 = nn.Conv1d(emb_dim, ch, kernel_size=7, padding=3)
        self.act = nn.ReLU()
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(ch * 3, n_labels)

    def forward(self, x):
        e = self.emb(x).transpose(1, 2)
        h3 = self.pool(self.act(self.conv3(e))).squeeze(-1)
        h5 = self.pool(self.act(self.conv5(e))).squeeze(-1)
        h7 = self.pool(self.act(self.conv7(e))).squeeze(-1)
        return self.fc(torch.cat([h3, h5, h7], dim=1))

model = CNNClassifier(n_labels=n_labels).to(device)

freq = train_terms_std[train_terms_std["protein_id"].isin(train_ids)]["go_id"].value_counts()
pos = np.array([freq.get(go, 0) for go in go_vocab], dtype=np.float32)
N = float(len(train_ids))

pos_weight = (N - pos) / (pos + 1e-6)
pos_weight = np.clip(pos_weight, 1.0, 50.0)  # cap is important
pos_weight_t = torch.tensor(pos_weight, dtype=torch.float32, device=device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scaler = GradScaler("cuda", enabled=(device.type == "cuda"))

print("pos_weight min/mean/max:", float(pos_weight_t.min()), float(pos_weight_t.mean()), float(pos_weight_t.max()))


model.train()
for ep in range(1, EPOCHS + 1):
    running = 0.0
    for step, (X, Y) in enumerate(train_loader, start=1):
        X = X.to(device, non_blocking=True)
        Y = Y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=(device.type == "cuda")):
            logits = model(X)
            loss = criterion(logits, Y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running += float(loss.item())
        if step % 200 == 0:
            print(f"epoch {ep} step {step}/{len(train_loader)} loss {running/200:.4f}")
            running = 0.0

MODEL_PATH = "/kaggle/working/cnn_go_model_balanced.pt"
torch.save(model.state_dict(), MODEL_PATH)
print("Saved model:", MODEL_PATH)

device: cuda
pos_weight min/mean/max: 1.4380220174789429 49.982025146484375 50.0
epoch 1 step 200/580 loss 0.0567
epoch 1 step 400/580 loss 0.0480
epoch 2 step 200/580 loss 0.0450
epoch 2 step 400/580 loss 0.0435
epoch 3 step 200/580 loss 0.0383
epoch 3 step 400/580 loss 0.0379
Saved model: /kaggle/working/cnn_go_model_balanced.pt


In [8]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast

small_test = test.head(5000).copy()

class SmallTestDataset(Dataset):
    def __init__(self, df):
        self.pids = df["protein_id"].astype(str).tolist()
        self.X = np.stack([encode_fixed(s) for s in df["sequence"].tolist()], axis=0)

    def __len__(self): 
        return len(self.pids)

    def __getitem__(self, idx):
        return self.pids[idx], self.X[idx]

def collate_small(batch):
    pids, xs = zip(*batch)
    X = torch.from_numpy(np.stack(xs, axis=0)).long()
    return list(pids), X

small_loader = DataLoader(
    SmallTestDataset(small_test),
    batch_size=256,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_small
)

TOPK = 200
all_go = []

model.eval()
with torch.no_grad():
    for pids, X in small_loader:
        X = X.to(device, non_blocking=True)
        with autocast("cuda", enabled=(device.type=="cuda")):
            prob = torch.sigmoid(model(X))

        _, idx = torch.topk(prob, k=TOPK, dim=1)
        idx = idx.cpu().numpy()

        for i in range(idx.shape[0]):
            all_go.extend([i2go[int(j)] for j in idx[i]])

print("unique GO in sample:", len(set(all_go)))

unique GO in sample: 8140


Trying to improve score

In [9]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class TestDataset(Dataset):
    def __init__(self, df):
        self.pids = df["protein_id"].astype(str).tolist()
        self.X = np.stack([encode_fixed(s) for s in df["sequence"].tolist()], axis=0)

    def __len__(self):
        return len(self.pids)

    def __getitem__(self, idx):
        return self.pids[idx], self.X[idx]

def collate_test(batch):
    pids, xs = zip(*batch)
    X = torch.from_numpy(np.stack(xs, axis=0)).long()
    return list(pids), X

test_loader = DataLoader(
    TestDataset(test[["protein_id","sequence"]].copy()),
    batch_size=256,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_test
)

print("test_loader ready. batches:", len(test_loader), "| proteins:", len(test))

test_loader ready. batches: 877 | proteins: 224309


In [10]:
TOPK_SUB = 300
THR = 0.05
OUT_PATH = "/kaggle/working/submission_300_thr005.tsv"

model.eval()
written = 0

with open(OUT_PATH, "w", encoding="utf-8") as f:
    with torch.no_grad():
        for pids, X in test_loader:
            X = X.to(device, non_blocking=True)
            with autocast("cuda", enabled=(device.type=="cuda")):
                prob = torch.sigmoid(model(X))

            sc, idx = torch.topk(prob, k=TOPK_SUB, dim=1)
            sc = sc.cpu().numpy()
            idx = idx.cpu().numpy()

            for i, pid in enumerate(pids):
                mask = sc[i] >= THR
                js = np.where(mask)[0]
                for j in js:
                    go = i2go[int(idx[i, j])]
                    s = float(sc[i, j])
                    f.write(f"{pid}\t{go}\t{s:.6f}\n")
                    written += 1

print("Wrote:", OUT_PATH, "| rows:", written)

Wrote: /kaggle/working/submission_300_thr005.tsv | rows: 67175213


**Build GO parent graph from Build GO parent graph from**

In [11]:
import re
from collections import defaultdict, deque

OBO_PATH = "/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo"

ID_RE = re.compile(r"^id:\s+(GO:\d{7})")
IS_A_RE = re.compile(r"^is_a:\s+(GO:\d{7})")

parents = defaultdict(list)
all_terms = set()

cur = None
with open(OBO_PATH, "r", encoding="utf-8") as f:
    for line in f:
        m = ID_RE.match(line)
        if m:
            cur = m.group(1)
            all_terms.add(cur)
            continue
        m = IS_A_RE.match(line)
        if m and cur:
            parents[cur].append(m.group(1))

print("terms in obo:", len(all_terms), "| edges:", sum(len(v) for v in parents.values()))

terms in obo: 48101 | edges: 62410


**Compute ancestors (cached) and propagate scores**

In [12]:
import numpy as np

anc_cache = {}

def get_ancestors(term):
    if term in anc_cache:
        return anc_cache[term]
    seen = set()
    stack = parents.get(term, [])
    while stack:
        p = stack.pop()
        if p in seen:
            continue
        seen.add(p)
        stack.extend(parents.get(p, []))
    anc_cache[term] = list(seen)
    return anc_cache[term]

def propagate_scores(go_ids, scores):
    """
    go_ids: list of GO terms predicted for one protein
    scores: list of their scores
    returns dict {go_term: score} with ancestor propagation via max
    """
    out = {}
    for go, s in zip(go_ids, scores):
        if go not in out or s > out[go]:
            out[go] = s
        for anc in get_ancestors(go):
            if anc not in out or s > out[anc]:
                out[anc] = s
    return out

**Predict test + write submission.tsv**

In [13]:
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
import numpy as np
import torch

SUB_PATH = "/kaggle/working/submission.tsv"
TOPK_SUB = 300
THR = 0.05
BATCH_SIZE = 256
NUM_WORKERS = 2

class TestDataset(Dataset):
    def __init__(self, df):
        self.pids = df["protein_id"].astype(str).tolist()
        self.X = np.stack([encode_fixed(s) for s in df["sequence"].tolist()], axis=0)

    def __len__(self): return len(self.pids)
    def __getitem__(self, idx): return self.pids[idx], self.X[idx]

def collate_test(batch):
    pids, xs = zip(*batch)
    X = torch.from_numpy(np.stack(xs, axis=0)).long()
    return list(pids), X

test_loader = DataLoader(TestDataset(test[["protein_id","sequence"]].copy()),
                         batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_test)

model.eval()
written = 0
with open(SUB_PATH, "w", encoding="utf-8") as f:
    with torch.no_grad():
        for pids, X in test_loader:
            X = X.to(device, non_blocking=True)
            with autocast("cuda", enabled=(device.type=="cuda")):
                prob = torch.sigmoid(model(X))

            sc, idx = torch.topk(prob, k=TOPK_SUB, dim=1)
            sc = sc.cpu().numpy()
            idx = idx.cpu().numpy()

            for i, pid in enumerate(pids):
                go_ids = [i2go[int(j)] for j in idx[i]]
                scores = sc[i].tolist()

                prop = propagate_scores(go_ids, scores)

                items = [(go, s) for go, s in prop.items() if s >= THR]
                if not items:
                    continue
                items.sort(key=lambda x: x[1], reverse=True)
                items = items[:TOPK_SUB]

                for go, s in items:
                    f.write(f"{pid}\t{go}\t{s:.6f}\n")
                    written += 1


print("Wrote:", SUB_PATH, "| rows:", written)

Wrote: /kaggle/working/submission.tsv | rows: 67284212


**Validate format<!--  -->**

In [14]:
GO_RE = re.compile(r"^GO:\d{7}$")

def validate_go_only(path, max_lines_to_check=200000):
    bad = []
    n = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            n += 1
            if max_lines_to_check and n > max_lines_to_check:
                break
            parts = line.rstrip("\n").split("\t")
            if len(parts) != 3:
                bad.append((n, f"wrong_num_cols:{len(parts)}"))
                continue
            pid, go, score = parts
            if not GO_RE.match(go):
                bad.append((n, f"bad_go_id:{go}"))
                continue
            try:
                s = float(score)
                if not (0.0 < s <= 1.0):
                    bad.append((n, f"score_out_of_range:{s}"))
            except Exception:
                bad.append((n, f"score_not_float:{score}"))
    return n, bad

n_lines, problems = validate_go_only(SUB_PATH)
print("checked:", n_lines)
print("problems:", problems[:20])


checked: 200001
problems: []


In [15]:
import os, pandas as pd

SUB_PATH = "/kaggle/working/submission_300_thr005.tsv"

print("size MB:", os.path.getsize(SUB_PATH)/(1024**2))

df = pd.read_csv(SUB_PATH, sep="\t", header=None, names=["protein_id","go_id","score"])
print("rows:", len(df))
print("unique proteins:", df["protein_id"].nunique(), " / expected:", len(test))
print("unique GO:", df["go_id"].nunique())
print("avg rows/protein:", df.groupby("protein_id").size().mean())
print("score min/mean/max:", df["score"].min(), df["score"].mean(), df["score"].max())

size MB: 1734.8080015182495
rows: 67175213
unique proteins: 224309  / expected: 224309
unique GO: 15518
avg rows/protein: 299.4762269904462
score min/mean/max: 0.050049 0.2599010603702595 0.999512
